# Build Structured + Unstructured Data Pipeline with ChromaDB

Use this notebook to turn the reviewed external-source outputs into:
- structured datasets for analytics and backtesting
- unstructured document and chunk datasets for retrieval
- a persistent ChromaDB collection for semantic search


## Scope

This notebook is the bridge between source review and production pipeline design. It assumes the source-review notebook has already generated the latest reviewed CSVs.

### What this notebook builds
- `processed/structured/*.csv`
- `processed/unstructured/*.csv` and `*.jsonl`
- `vector/chroma/*` persistent ChromaDB storage

### Helpful terms
- `Structured dataset` = tables we can filter, join, and backtest
- `Unstructured dataset` = cleaned long-form text with metadata
- `Chunk` = a smaller text segment cut from a longer document
- `Embedding` = a numeric vector representation of text for semantic retrieval
- `ChromaDB` = a vector database that stores text embeddings and metadata


In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'data_pipeline' else Path.cwd().resolve()
DATA_DIR = ROOT / 'data'
EXTERNAL_DIR = DATA_DIR / 'external'
INTERIM_DIR = DATA_DIR / 'interim'
REVIEW_DIR = INTERIM_DIR / 'source_reviews'
PROCESSED_DIR = DATA_DIR / 'processed'
STRUCTURED_DIR = PROCESSED_DIR / 'structured'
UNSTRUCTURED_DIR = PROCESSED_DIR / 'unstructured'
VECTOR_DIR = DATA_DIR / 'vector'
CHROMA_DIR = VECTOR_DIR / 'chroma_portfolio_pipeline'

for path in [STRUCTURED_DIR, UNSTRUCTURED_DIR, VECTOR_DIR, CHROMA_DIR]:
    path.mkdir(parents=True, exist_ok=True)

def safe_read_csv(path: Path) -> pd.DataFrame:
    if path.exists():
        return pd.read_csv(path)
    print(f'Missing file: {path}')
    return pd.DataFrame()

def write_jsonl(df: pd.DataFrame, path: Path) -> None:
    records = df.to_dict(orient='records')
    with path.open('w', encoding='utf-8') as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + '\n')

def normalize_whitespace(text: str) -> str:
    return re.sub(r'\s+', ' ', str(text or '')).strip()

def safe_text(value: object) -> str:
    text = normalize_whitespace(str(value or ''))
    if text.lower().startswith('error:'):
        return ''
    return text

def chunk_text(text: str, *, chunk_size: int = 1200, overlap: int = 200) -> list[str]:
    text = normalize_whitespace(text)
    if not text:
        return []
    chunks: list[str] = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks

source_files = {
    'company_master': EXTERNAL_DIR / 'sec' / 'company_master_full_universe.csv',
    'filings_full': EXTERNAL_DIR / 'sec' / 'filings_full_universe.csv',
    'latest_reporting_filings': EXTERNAL_DIR / 'sec' / 'latest_reporting_filings.csv',
    'company_facts_full': EXTERNAL_DIR / 'sec' / 'company_facts_full.csv',
    'fact_tag_dictionary': EXTERNAL_DIR / 'sec' / 'fact_tag_dictionary.csv',
    'sec_document_text_full': REVIEW_DIR / 'sec_document_text_full.csv',
    'macro_series': EXTERNAL_DIR / 'fred' / 'macro_series_full_review.csv',
    'fmp_profiles': EXTERNAL_DIR / 'fmp' / 'company_profiles_full_universe.csv',
    'gdelt_articles': EXTERNAL_DIR / 'gdelt' / 'articles_full_universe.csv',
    'gdelt_fetch_log': REVIEW_DIR / 'gdelt_fetch_log.csv',
    'article_text_full': REVIEW_DIR / 'gdelt_article_text_full.csv',
}

source_files


## 1. Load Reviewed Source Outputs

We load the latest reviewed datasets from the earlier notebooks. If something is missing, the notebook will show it explicitly so we can decide whether to regenerate it first.


In [ ]:
datasets = {name: safe_read_csv(path) for name, path in source_files.items()}
source_availability = pd.DataFrame([
    {
        'dataset': name,
        'path': str(path),
        'exists': path.exists(),
        'rows': len(datasets[name]),
        'columns': len(datasets[name].columns),
    }
    for name, path in source_files.items()
])
display(source_availability)


## 2. Build Structured Datasets

These are the datasets we would store in a structured layer for analytics, filtering, joins, backtests, and later object construction.


In [ ]:
company_master = datasets['company_master'].copy()
filings_full = datasets['filings_full'].copy()
latest_reporting_filings = datasets['latest_reporting_filings'].copy()
company_facts_full = datasets['company_facts_full'].copy()
fact_tag_dictionary = datasets['fact_tag_dictionary'].copy()
macro_series = datasets['macro_series'].copy()
fmp_profiles = datasets['fmp_profiles'].copy()
gdelt_articles = datasets['gdelt_articles'].copy()
gdelt_fetch_log = datasets['gdelt_fetch_log'].copy()

structured_companies = company_master.copy()

structured_filings = latest_reporting_filings.copy()
if not structured_filings.empty:
    keep_cols = [c for c in ['ticker', 'company_name', 'cik', 'form', 'filingDate', 'accessionNumber', 'primaryDocument', 'primary_document_url'] if c in structured_filings.columns]
    structured_filings = structured_filings[keep_cols]

structured_company_facts = company_facts_full.copy()
if not structured_company_facts.empty and 'priority_tier' in structured_company_facts.columns:
    structured_company_facts = structured_company_facts.sort_values(['ticker', 'priority_tier', 'fact_tag', 'filed_date'])

structured_macro = macro_series.copy()
structured_profiles = fmp_profiles.copy()
structured_news_metadata = gdelt_articles.copy()
structured_news_fetch_log = gdelt_fetch_log.copy()

structured_outputs = {
    'companies.csv': structured_companies,
    'filings.csv': structured_filings,
    'company_facts.csv': structured_company_facts,
    'fact_tag_dictionary.csv': fact_tag_dictionary,
    'macro_series.csv': structured_macro,
    'company_profiles.csv': structured_profiles,
    'news_metadata.csv': structured_news_metadata,
    'news_fetch_log.csv': structured_news_fetch_log,
}

for filename, frame in structured_outputs.items():
    frame.to_csv(STRUCTURED_DIR / filename, index=False)

structured_summary = pd.DataFrame([
    {'output_file': filename, 'rows': len(frame), 'columns': len(frame.columns)}
    for filename, frame in structured_outputs.items()
])
display(structured_summary)


## 3. Build Unstructured Document Corpus

These are the long-form documents that later feed chunking, embeddings, and retrieval. We only keep cleaned, reviewable text here.


In [ ]:
sec_document_text_full = datasets['sec_document_text_full'].copy()
article_text_full = datasets['article_text_full'].copy()

sec_docs = pd.DataFrame()
if not sec_document_text_full.empty:
    sec_docs = pd.DataFrame({
        'document_id': [f"sec::{ticker}::{form}::{date}" for ticker, form, date in zip(sec_document_text_full.get('ticker', []), sec_document_text_full.get('form', []), sec_document_text_full.get('filing_date', []))],
        'source_type': 'sec_filing',
        'ticker': sec_document_text_full.get('ticker', ''),
        'domain': 'sec.gov',
        'title': sec_document_text_full.get('form', '').astype(str) + ' filing',
        'document_date': sec_document_text_full.get('filing_date', ''),
        'url': sec_document_text_full.get('primary_document_url', ''),
        'body_text': sec_document_text_full.get('selected_clean_text', '').map(safe_text),
        'metadata_json': sec_document_text_full.apply(lambda row: json.dumps({
            'ticker': row.get('ticker'),
            'source_type': 'sec_filing',
            'form': row.get('form'),
            'filing_date': row.get('filing_date'),
            'url': row.get('primary_document_url'),
        }), axis=1),
    })
    sec_docs = sec_docs[sec_docs['body_text'].astype(bool)]

news_docs = pd.DataFrame()
if not article_text_full.empty:
    news_docs = pd.DataFrame({
        'document_id': [f"news::{ticker}::{idx}" for idx, ticker in enumerate(article_text_full.get('ticker', []))],
        'source_type': 'news_article',
        'ticker': article_text_full.get('ticker', ''),
        'domain': article_text_full.get('domain', ''),
        'title': article_text_full.get('title', ''),
        'document_date': article_text_full.get('seendate', ''),
        'url': article_text_full.get('url', ''),
        'body_text': article_text_full.get('selected_text', '').map(safe_text),
        'metadata_json': article_text_full.apply(lambda row: json.dumps({
            'ticker': row.get('ticker'),
            'source_type': 'news_article',
            'domain': row.get('domain'),
            'seendate': row.get('seendate'),
            'url': row.get('url'),
            'title': row.get('title'),
        }), axis=1),
    })
    news_docs = news_docs[news_docs['body_text'].astype(bool)]

document_corpus = pd.concat([sec_docs, news_docs], ignore_index=True) if not sec_docs.empty or not news_docs.empty else pd.DataFrame(columns=['document_id', 'source_type', 'ticker', 'domain', 'title', 'document_date', 'url', 'body_text', 'metadata_json'])
document_corpus.to_csv(UNSTRUCTURED_DIR / 'document_corpus.csv', index=False)
write_jsonl(document_corpus, UNSTRUCTURED_DIR / 'document_corpus.jsonl')
display(document_corpus.head(20))
print('document count:', len(document_corpus))


## 4. Chunk the Unstructured Corpus

Chunks are what we actually embed and store in ChromaDB. Each chunk keeps enough metadata to stay traceable back to the original document.


In [ ]:
chunk_rows = []
for row in document_corpus.itertuples(index=False):
    chunks = chunk_text(row.body_text, chunk_size=1200, overlap=200)
    for idx, chunk in enumerate(chunks):
        chunk_rows.append({
            'chunk_id': f'{row.document_id}::chunk_{idx:03d}',
            'document_id': row.document_id,
            'source_type': row.source_type,
            'ticker': row.ticker,
            'domain': row.domain,
            'title': row.title,
            'document_date': row.document_date,
            'url': row.url,
            'chunk_index': idx,
            'chunk_text': chunk,
        })

chunk_df = pd.DataFrame(chunk_rows)
chunk_df.to_csv(UNSTRUCTURED_DIR / 'document_chunks.csv', index=False)
write_jsonl(chunk_df, UNSTRUCTURED_DIR / 'document_chunks.jsonl')

chunk_summary = pd.DataFrame([
    {'source_type': source_type, 'chunks': len(frame), 'tickers': frame['ticker'].nunique()}
    for source_type, frame in chunk_df.groupby('source_type')
]) if not chunk_df.empty else pd.DataFrame()
display(chunk_summary)
display(chunk_df.head(20))


## 5. Build the ChromaDB Collection

This cell persists the chunks into a ChromaDB collection. It is intentionally explicit so you can inspect how many chunks went in and what metadata was attached.

If `chromadb` is not installed in this kernel yet, install it first, for example:
```python
%pip install chromadb
```


In [ ]:
try:
    import chromadb
    from chromadb.utils import embedding_functions
except ImportError as error:
    raise ImportError('Install chromadb in this notebook kernel before running this cell: %pip install chromadb') from error

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection_name = 'portfolio_external_intelligence'
existing_collections = {collection.name for collection in client.list_collections()}
if collection_name in existing_collections:
    client.delete_collection(collection_name)

embedding_fn = embedding_functions.DefaultEmbeddingFunction()
collection = client.create_collection(
    name=collection_name,
    embedding_function=embedding_fn,
    metadata={'hnsw:space': 'cosine'},
)

if not chunk_df.empty:
    batch_size = 100
    for start in range(0, len(chunk_df), batch_size):
        batch = chunk_df.iloc[start:start + batch_size]
        metadatas = [
            {
                'document_id': str(row.document_id),
                'source_type': str(row.source_type),
                'ticker': str(row.ticker),
                'domain': str(row.domain),
                'title': str(row.title),
                'document_date': str(row.document_date),
                'url': str(row.url),
                'chunk_index': int(row.chunk_index),
            }
            for row in batch.itertuples(index=False)
        ]
        collection.add(
            ids=batch['chunk_id'].astype(str).tolist(),
            documents=batch['chunk_text'].astype(str).tolist(),
            metadatas=metadatas,
        )

print('chroma path:', CHROMA_DIR)
print('collection:', collection_name)
print('chunk count:', collection.count())


## 6. Example Retrieval Query

This is a quick smoke test to confirm the vector store is usable for semantic retrieval.


In [ ]:
if 'collection' not in globals():
    print('Run the ChromaDB setup cell first.')
elif collection.count() == 0:
    print('Collection is empty. Build the chunk corpus first.')
else:
    query = 'What recent AI infrastructure commentary matters for Amazon?'
    results = collection.query(query_texts=[query], n_results=5)
    result_rows = []
    for idx in range(len(results['ids'][0])):
        result_rows.append({
            'rank': idx + 1,
            'chunk_id': results['ids'][0][idx],
            'distance': results['distances'][0][idx] if 'distances' in results else None,
            'document': results['documents'][0][idx],
            **results['metadatas'][0][idx],
        })
    result_df = pd.DataFrame(result_rows)
    display(result_df[['rank', 'ticker', 'source_type', 'domain', 'title', 'distance', 'document']])


## Notes

Recommended architecture after this notebook:
- keep structured datasets in `data/processed/structured`
- keep document and chunk corpora in `data/processed/unstructured`
- keep ChromaDB persistence under `data/vector/chroma_portfolio_pipeline`
- later, load these rows into validated objects before handing them to the LLM / reasoning layer
